# Open-Meteo - warstwa BRONZE

Cel: zamiana plików JSON z pogodą (jeden na stację mierzącą PM2.5) na jedną tabelę i zapis do Parquet.

Źródło: Open-Meteo Historical Weather API, licencja CC BY 4.0.

## 1. Ścieżki i pliki źródłowe

Wczytujemy listę plików JSON pobranych w notebooku 01

In [1]:
from pathlib import Path
import json

import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.pipeline_config import YEARS, PERIOD, START, END, HOURS
from scripts import bronze_openmeteo as openmeteo_module
paths = openmeteo_module.weather_files()
print('Plików JSON:', len(paths))

Plików JSON: 381


## 2. Jak wygląda jedna odpowiedź

Odpowiedź zawiera opis punktu (współrzędne, wysokość, jednostki) oraz część `hourly`, w której każda zmienna to lista wartości godzinowych. Czas jest w UTC.

In [2]:
payload = json.loads(paths[0].read_text(encoding='utf-8'))

print({key: value for key, value in payload.items() if key != 'hourly'})
print()

for name, values in payload['hourly'].items():
    print(f'{name:22} {len(values)} wartości, np. {values[:3]}')

{'latitude': 51.28295, 'longitude': 15.8918915, 'generationtime_ms': 96.67730331420898, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 152.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'surface_pressure': 'hPa', 'wind_speed_10m': 'km/h', 'wind_direction_10m': '°'}}

time                   8760 wartości, np. ['2023-01-01T00:00', '2023-01-01T01:00', '2023-01-01T02:00']
temperature_2m         8760 wartości, np. [16.1, 16.7, 17.4]
relative_humidity_2m   8760 wartości, np. [54, 51, 47]
precipitation          8760 wartości, np. [0.0, 0.0, 0.0]
surface_pressure       8760 wartości, np. [997.2, 997.3, 997.5]
wind_speed_10m         8760 wartości, np. [27.5, 27.5, 28.0]
wind_direction_10m     8760 wartości, np. [236, 236, 241]


## 3. Konwersja wszystkich plików

Funkcja `json_to_bronze` ze `scripts/bronze_openmeteo.py` zamienia jeden plik na tabelę, w której jeden wiersz to jedna stacja i jedna godzina. Kod stacji bierzemy z nazwy pliku, czas zamieniamy na datę, a zmienne pogodowe na liczby.

In [3]:
from scripts import bronze_openmeteo as openmeteo_module
weather = openmeteo_module.main()

print(weather.schema)
print(weather.head(3))

Pliki JSON: 381
RAZEM: (3340608, 12)
Zapisano: data/bronze/openmeteo_2023_2025.parquet (29.3 MB)
Schema({'kod_stacji': String, 'data_czas_utc': Datetime(time_unit='us', time_zone=None), 'temperature_2m': Float64, 'relative_humidity_2m': Float64, 'precipitation': Float64, 'surface_pressure': Float64, 'wind_speed_10m': Float64, 'wind_direction_10m': Float64, 'siatka_szerokosc': Float64, 'siatka_dlugosc': Float64, 'wysokosc_m': Float64, 'plik_zrodlowy': String})
shape: (3, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ kod_stacj ┆ data_czas ┆ temperatu ┆ relative_ ┆ … ┆ siatka_sz ┆ siatka_dl ┆ wysokosc_ ┆ plik_zro │
│ i         ┆ _utc      ┆ re_2m     ┆ humidity_ ┆   ┆ erokosc   ┆ ugosc     ┆ m         ┆ dlowy    │
│ ---       ┆ ---       ┆ ---       ┆ 2m        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ datetime[ ┆ f64       ┆ ---       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ str      │
│           ┆ μs

## 4. Kontrola

Każda stacja z sumy roczników ma pełną siatkę godzin 2023–2025. Liczba godzin wynika z zakresu dat i uwzględnia rok przestępny.

In [4]:
print('wiersze:', weather.height)
print('stacje:', weather['kod_stacji'].n_unique())
print('zakres czasu:', weather['data_czas_utc'].min(), '-', weather['data_czas_utc'].max())
print('duplikaty (stacja, godzina):', weather.select('kod_stacji', 'data_czas_utc').is_duplicated().sum())
print()
print(weather.null_count())
print(weather.select(openmeteo_module.WEATHER_COLUMNS).describe())
assert weather.group_by("kod_stacji").len()["len"].unique().to_list() == [HOURS]


wiersze: 3340608
stacje: 127
zakres czasu: 2023-01-01 00:00:00 - 2025-12-31 23:00:00
duplikaty (stacja, godzina): 0

shape: (1, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ kod_stacj ┆ data_czas ┆ temperatu ┆ relative_ ┆ … ┆ siatka_sz ┆ siatka_dl ┆ wysokosc_ ┆ plik_zro │
│ i         ┆ _utc      ┆ re_2m     ┆ humidity_ ┆   ┆ erokosc   ┆ ugosc     ┆ m         ┆ dlowy    │
│ ---       ┆ ---       ┆ ---       ┆ 2m        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ u32       ┆ u32       ┆ u32       ┆ ---       ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32      │
│           ┆           ┆           ┆ u32       ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0         ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0        │
└───────────┴───────────┴───────────┴───────────┴───┴───────

shape: (9, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ temperature_ ┆ relative_hum ┆ precipitati ┆ surface_pre ┆ wind_speed_ ┆ wind_direct │
│ ---        ┆ 2m           ┆ idity_2m     ┆ on          ┆ ssure       ┆ 10m         ┆ ion_10m     │
│ str        ┆ ---          ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│            ┆ f64          ┆ f64          ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 3.340608e6   ┆ 3.340608e6   ┆ 3.340608e6  ┆ 3.340608e6  ┆ 3.340608e6  ┆ 3.340608e6  │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 10.344666    ┆ 76.28498     ┆ 0.084609    ┆ 996.691517  ┆ 12.706884   ┆ 200.251681  │
│ std        ┆ 8.333639     ┆ 16.866361    ┆ 0.382887    ┆ 14.134411   ┆ 6.57